# Paper 17: Point-in-time universe effect

Measure how much of the v1 path came from survivorship (today's S&P 500 list) versus the point-in-time members in v2 baseline. Current-list is the leak, not a live candidate — keep PIT even if it looks better.

Scored against **v2 baseline**. Kill rules are frozen in `Research/book-construction-status.md`:

- Train 2022 calendar more than 5pp worse than SPUS
- Train max drawdown worse than SPUS
- Train Calmar below both v2 baseline and SPUS
- Test 2023 calendar more than 15pp worse than v2 baseline

**Variants.** v2-baseline (PIT) vs current-list (pit=false).


In [ ]:
from pathlib import Path

from monterey.diagnostics.performance import yearly
from monterey.research import boot, compare
from monterey.sprints import (
    run_sprint,
    score_against_baseline,
    survivorship_in_book,
    universe_survivorship,
    v2_snapshot_hash,
)

rd = boot("paper-17-pit-universe", snapshot=v2_snapshot_hash())
print("snapshot", rd.manifest.get("hash"))
census = universe_survivorship(rd)
print(census.agg({"n_pit": "mean", "n_current": "mean", "joiners": "mean", "leavers": "mean"}))
census.tail()


In [ ]:
ledgers, table = run_sprint("17", rd)
table


In [ ]:
baseline, current = ledgers
ghosts = survivorship_in_book(current, rd)
leavers = survivorship_in_book(baseline, rd)
joiners = ghosts[ghosts["kind"] == "joiner"] if not ghosts.empty else ghosts
gone = leavers[leavers["kind"] == "leaver"] if not leavers.empty else leavers
print("current-list names held before they joined the index")
print(joiners.groupby("symbol")["weight"].agg(["count", "mean", "max"]).sort_values("max", ascending=False).head(20) if not joiners.empty else joiners)
print("PIT names held after they left today's list")
print(gone.groupby("symbol")["weight"].agg(["count", "mean", "max"]).sort_values("max", ascending=False).head(20) if not gone.empty else gone)


In [ ]:
import matplotlib.pyplot as plt

figures = Path("figures")
figures.mkdir(exist_ok=True)
fig, axes = plt.subplots(2, 1, figsize=(10, 8))
axes[0].plot(census["date"], census["n_pit"], label="PIT members")
axes[0].plot(census["date"], census["n_current"], label="today's list")
axes[0].set_title("S&P 500 membership: point-in-time vs end-date list")
axes[0].legend()
for book in ledgers:
    nav = book.nav_series()
    axes[1].plot(nav.index, nav / nav.iloc[0], label=book.spec.id)
spus = rd.total_return("SPUS").reindex(ledgers[0].nav_series().index).ffill()
axes[1].plot(spus.index, spus / spus.iloc[0], label="SPUS", alpha=0.7)
axes[1].set_title("NAV, start = 1")
axes[1].legend()
fig.tight_layout()
fig.savefig(figures / "pit-vs-current.png")
plt.close(fig)
print("wrote", figures / "pit-vs-current.png")
print(yearly(ledgers[0].nav_series()).join(yearly(ledgers[1].nav_series()), lsuffix="_pit", rsuffix="_current"))


In [ ]:
verdict = score_against_baseline(current, baseline, rd)
print(current.spec.id, "KILLED" if verdict["killed"] else "looks fine on kill bars", "; ".join(verdict["reasons"]) or "pass")
print("Architecture: keep universe.pit=true. current-list is survivorship, not a live rule.")
